# 05 | SQL Database Design, Integrity & Analytical Views

**Project:** Dubai Short-Term Rental Intelligence  
**Market / source:** Dubai, AirROI historical short-term rental sample (distributed via Kaggle)  
**Observed coverage:** March 2021–February 2026  
**Stack:** Python · pandas · SQLite · SQL · Power BI

### Business objective

Provide a reproducible relational foundation for comparing property-month performance, identifying potential pricing-review cases, and delivering auditable inputs to Power BI. The SQL layer must maintain traceability to the transformed records produced in Notebooks 03 and 04.

**Scope and interpretation:** This is a third-party sample, not a census of Dubai rentals or internal Airbnb transactions. The supplier's occupancy and revenue measures are estimates. Pricing-review flags indicate candidates for investigation, **not** optimal-price or causal revenue recommendations.

## 1. Reproducibility and inputs

This notebook requires six processed CSVs generated by the preceding notebooks: `dim_listing`, `dim_month`, `fact_listing_month`, `fact_coverage_month`, `fact_peer_benchmark`, and `peer_examples_latest`. The SQL builder is maintained in `src/03_build_powerbi_exports.py`; views are defined in `sql/02_analytics_views.sql`. Source data is **not** modified.

**Rebuild policy:** The script validates a new SQLite database in a temporary location. Only a successful build replaces the previous SQLite file. Power BI CSV exports are staged and published after successful database validation. Row counts are reconciled dynamically with the current processed inputs rather than hardcoded to the 2026 snapshot.

**Dependencies:** `pandas`, `numpy`, `IPython`, standard-library `sqlite3`. See `requirements.txt` for the project environment.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import json
import runpy
import sqlite3
import pandas as pd
from IPython.display import Markdown, display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CWD, *CWD.parents]
                     if (p / "src" / "03_build_powerbi_exports.py").is_file()
                     and (p / "sql" / "02_analytics_views.sql").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project directory or notebooks/ folder.")
PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_SQL = PROJECT_ROOT / "reports" / "sql"
REPORT_SQL.mkdir(parents=True, exist_ok=True)

required = ["dim_listing", "dim_month", "fact_listing_month",
            "fact_coverage_month", "fact_peer_benchmark", "peer_examples_latest"]
missing = [name for name in required if not (PROCESSED / f"{name}.csv").is_file()]
if missing:
    raise FileNotFoundError(f"Processed inputs missing: {missing}. Run Notebooks 03–04 first.")
print("Project:", PROJECT_ROOT.name)
print("Processed files available:", len(required))

## 2. Relational architecture and key design

Each table has an explicitly declared primary key and relevant foreign keys. All listing identifiers use `TEXT` to avoid precision loss from long numeric-looking identifiers.

| Table | Business grain / primary key | Relationships | Analytical role |
|---|---|---|---|
| `dim_listing` | One row per listing; `listing_id` | Parent of monthly listing facts | Property attributes (latest available; not historically versioned) |
| `dim_month` | One row per month; `month_date` | Parent of monthly facts and coverage | Reporting calendar |
| `fact_listing_month` | One listing and month; `(listing_id, month_date)` | FK → listing, FK → month | Historical property-month metrics |
| `fact_coverage_month` | One row per reporting month; `month_date` | FK → month | Sample-size and completeness monitoring |
| `fact_peer_benchmark` | One listing and month; `(listing_id, month_date)` | Composite FK → listing-month fact | Competitor-benchmark results |
| `peer_examples_latest` | One subject–peer pair per month; `(listing_id, month_date, peer_listing_id)` | FKs → subject fact and peer fact | Auditable selected examples of matched competitors |

**Key distinction:** `peer_examples_latest` contains *examples*, not the complete competitor set used to calculate a benchmark. `dim_listing` provides the latest available attributes, which may not represent a property's historical characteristics for every month.

SQLite enforces the declared constraints only when `PRAGMA foreign_keys=ON` is enabled on each connection. The builder enables it *before* loading any rows. Power BI itself does not enforce SQLite foreign keys; the model's relationships must be configured separately. **When importing CSVs into Power BI, explicitly assign `listing_id` and `peer_listing_id` the Text data type** before Power Query or model relationships can coerce long IDs into rounded numeric values.

## 3. Safe, constraint-enforced database build

The builder checks source identifiers and boolean fields, creates declared SQLite tables rather than allowing inferred `to_sql()` schemas, loads dimensions before facts, installs the reusable analytical views, reconciles record counts, checks SQL view logic, and tests that an invalid foreign-key insert is rejected. Failed builds leave the previous database untouched.

In [2]:
make_database = runpy.run_path(
    str(PROJECT_ROOT / "src" / "03_build_powerbi_exports.py")
)["run"]

build_audit = make_database()
db_path = PROCESSED / "dubai_rentals.sqlite"
assert db_path.is_file()

record_counts = pd.DataFrame(
    [{"table": table, "validated_rows": count}
     for table, count in build_audit["table_counts"].items()]
)
display(record_counts)
print("Latest month:", build_audit["latest_month"])
print("Database:", db_path.relative_to(PROJECT_ROOT))

## 4. Schema, constraints and relationship verification

The audit below inspects SQLite's actual catalog, not just the intended schema. It verifies declared composite/single-column primary keys, existing foreign-key references, enabled enforcement, text-based listing identifiers, and SQLite's structural integrity check.

In [3]:
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys=ON")
assert conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1
assert conn.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
assert conn.execute("PRAGMA foreign_key_check").fetchall() == []

schema_audit = []
for table in required:
    cols = conn.execute(f'PRAGMA table_info("{table}")').fetchall()
    pk = ", ".join(row[1] for row in sorted(cols, key=lambda r: r[5]) if row[5] > 0)
    foreign_keys = conn.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
    schema_audit.append({
        "table": table,
        "columns": len(cols),
        "primary_key": pk,
        "foreign_key_columns": len(foreign_keys),
        "records": conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0],
    })
schema_audit = pd.DataFrame(schema_audit)
display(schema_audit)
assert (schema_audit.primary_key.str.len() > 0).all()
assert conn.execute("SELECT DISTINCT typeof(listing_id) FROM fact_listing_month").fetchall() == [("text",)]
print("Enforced foreign keys: ON | Violations: 0 | SQLite integrity: ok")

## 5. Source reconciliation and meaningful analytical-view tests

A successful SQL statement does not guarantee a correct business calculation. The database builder therefore independently compares: all six imported row counts with processed inputs; latest pricing-review flags with the stored benchmark file; segment counts against active source listing-month rows; monthly coverage against the source coverage table; benchmark-status totals against the latest snapshot; and consecutive-month observations against a pandas recomputation.

The notebook separately displays each SQL view and confirms that the validated results are still queryable. The latest review queue is also compared record-by-record for listing IDs, nightly rates, peer medians, premium percentages, peer counts and radii; latest segment mean rates and listing counts are cross-checked against pandas aggregations.

In [4]:
view_names = pd.read_sql_query("""
SELECT name FROM sqlite_master WHERE type='view' ORDER BY name
""", conn)["name"].tolist()
view_audit = pd.DataFrame([
    {"sql_view": name,
     "returned_rows": conn.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]}
    for name in view_names
])
display(view_audit)
assert {'vw_market_monthly','vw_listing_rate_transitions','vw_fixed_cohort_monthly','vw_segment_pricing','vw_latest_review_priority'} <= set(view_names)
assert build_audit["review_queue"] == int(conn.execute(
    "SELECT COUNT(*) FROM vw_latest_price_review_queue").fetchone()[0])
assert build_audit["same_listing_month_pairs"] == int(conn.execute(
    "SELECT COUNT(*) FROM vw_pairwise_same_listing_month").fetchone()[0])
print("Independently reconciled latest pricing review queue:", build_audit["review_queue"])
print("Independently reconciled consecutive-month same-listing pairs:",
      build_audit["same_listing_month_pairs"])

## 6. Traceability example: pricing-review queue

`vw_latest_price_review_queue` joins the benchmark output to the corresponding listing-month observation and returns properties flagged by the transparent screening rules developed in Notebook 04. These records should prompt follow-up investigation, not automatic price changes. The view includes the peer count, comparison radius and confidence label so users can assess benchmark limitations.

In [5]:
queue_sample = pd.read_sql_query("""
SELECT listing_id, month_date, room_type, bedrooms,
       ROUND(subject_adr_aed, 1) AS subject_adr_aed,
       peer_median_aed, premium_pct, peer_count,
       radius_km, review_signal, confidence_label
FROM vw_latest_price_review_queue
ORDER BY ABS(premium_pct) DESC, listing_id
LIMIT 5
""", conn)
print('Private traceability query returned', len(queue_sample), 'rows; listing-level results are intentionally not published.')
review_summary = pd.read_sql_query("""
SELECT review_signal, COUNT(*) AS flagged_listings
FROM vw_latest_price_review_queue
GROUP BY review_signal ORDER BY flagged_listings DESC
""", conn)
display(review_summary)
assert int(review_summary.flagged_listings.sum()) == build_audit["review_queue"]

## 7. Validation report and reproducible handoff

An audit record is written to `reports/sql/database_build_validation.json`. This captures the dataset-specific row counts, the latest observation month, structural integrity checks, and the business-view reconciliation results. Publishing the audit provides evidence of testing without uploading the raw third-party dataset or the generated SQLite database.

In [6]:
validation_output = REPORT_SQL / "database_build_validation.json"
validation_output.write_text(json.dumps(build_audit, indent=2), encoding="utf-8")
print("Saved:", validation_output.relative_to(PROJECT_ROOT))
conn.close()

## 8. Key findings and analytical decisions

**Database implementation.** The current processed dataset contains **99,745 listing-month observations**, **4,364 listings**, and **60 reporting months**. The SQL database contains **six tables**, including the property-month benchmark and a traceable set of competitor examples. **Ten analytical views** support reporting, historical comparisons, and pricing-review investigations.

**Integrity and reconciliation.** All six table counts match their processed source files. Every table has a declared primary key; related tables have enforced foreign-key constraints. SQLite's integrity check and `foreign_key_check` return no violations, and a deliberately invalid foreign-key insertion is rejected. Business-view tests reconcile the latest **902 review-flagged listings** against the benchmark records, as well as coverage, segmentation, statuses, and consecutive-month comparisons.

**Business interpretation.** The review queue is a structured screening tool for property managers, not a pricing optimization engine. Benchmarks use limited comparability attributes, and occupancy/revenue values are supplier estimates. The data describes the observed AirROI sample, not Dubai's entire market.

**Technical decisions.** Listing IDs are stored as text; row-count checks are calculated from source inputs; the database is built in a temporary file and published only after validation. The monthly dimension and listing dimension can be related to the two primary fact tables in Power BI. Identifier fields must remain Text during the Power BI import; `host_id` is preserved as text from source. `metadata_source` identifies the undated Parquet snapshot, and `native_booked_rate_avg` is a separate nullable provider metric. The latest-attributes listing dimension is not a historical slowly changing dimension.

**Next stage.** Notebook 06 will perform business analysis through the validated SQL tables and views. Any new reporting metric should be reconciled to its source-level definition before inclusion in Power BI.